# Tune the bootstrap-PF gains with Optuna

The no-network baseline (`use_net=False`) replaces the 13 controller outputs with
three fixed gains: process-noise scale, prior-regularization weight, and likelihood
temperature. Here we search for good **constant** values with Optuna, minimizing the
leave-one-unit-out dev tail-NLL (same metric as training). Trials are independent and
CPU-bound, so they parallelize across cores via `n_jobs`.

In [1]:
import json
import os

import numpy as np
import optuna
import pandas as pd
import torch

from experiment_config import SEED, DegModel, dataset_paths, pfnet_paths
from src.helpers.seed import set_global_seed
from src.models.particle_filter.core import ParticleFilter
from src.training.pfnet_hparams import PFNET_ARGS


## Configuration

In [2]:
DATA_NAME = "DS03"
EVAL_REPS = 3
N_TRIALS = 200

In [3]:
ARGS_ID = 0

# PF + eval settings come from the shared config so tuning matches the pipeline
_args = PFNET_ARGS[ARGS_ID]
N_PARTICLES = int(_args["PARTICLE_FILTER"]["N_PARTICLES"])
MAX_LIFE = int(_args["PARTICLE_FILTER"]["MAX_LIFE"])
LOSS_TAIL_STEPS = int(_args["TRAINING"]["LOSS_TAIL_STEPS"])
SEED_STRIDE = int(_args["EVALUATION"]["SEED_STRIDE"])

# one torch thread per Optuna worker so threads do not oversubscribe the cores
torch.set_num_threads(1)
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK") or os.cpu_count() or 1)

ESTIMATION_DIR, DEGR_MODEL_DIR = dataset_paths(
    DATA_NAME, fields=["estimation", "degr_model"]
)
_, PRED_DIR = pfnet_paths(ARGS_ID, DATA_NAME)
PRED_DIR.mkdir(parents=True, exist_ok=True)
set_global_seed(SEED)
print(f"n_jobs={N_JOBS}")


n_jobs=16


## Load dev data and degradation models (once)

In [4]:
dev_hi = pd.read_csv(ESTIMATION_DIR / "data_dev.csv")
dev_units = sorted(dev_hi["unit"].astype(int).unique().tolist())
perform_names = [c for c in dev_hi.columns if c not in ["unit", "cycle", "hs"]]


def build_tensors(df, name):
    out = {}
    for u in dev_units:
        sub = df[df["unit"] == u]
        out[u] = torch.tensor(
            np.stack([sub["cycle"].values, sub[name].values], axis=1),
            dtype=torch.float32,
        )
    return out


dev_tensors, dev_degmodels = {}, {}
for name in perform_names:
    dev_tensors[name] = build_tensors(dev_hi, name)
    models = {}
    for u in dev_units:
        m = DegModel()
        m.load_state_dict(
            torch.load(DEGR_MODEL_DIR / "states" / name / f"unit_{u}" / "best_model.pt")
        )
        models[u] = m
    dev_degmodels[name] = models

print("dev units:", dev_units, "| metrics:", perform_names)

dev units: [1, 2, 3, 4, 5, 6, 7, 8, 9] | metrics: ['T48', 'SmFan', 'SmLPC', 'SmHPC']


## Tail-NLL evaluation with tunable gains

In [5]:
def tail_nll(pf, t_data, s_data):
    step_losses = []
    for k in range(len(t_data)):
        mixture = pf.step(t_obs=t_data[[k]], s_obs=s_data[[k]])
        start = -LOSS_TAIL_STEPS if LOSS_TAIL_STEPS else k
        dist = mixture.distribution(s=s_data[start:])
        step_losses.append(-dist.log_prob(t_data[start:]).mean())
    return float(torch.stack(step_losses).mean().item())


@torch.no_grad()
def eval_unit(base_models, unit_tensor, seeds, cn, cp, cl):
    t_data, s_data = unit_tensor[:, 0], unit_tensor[:, 1]
    reps = []
    for sd in seeds:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(sd)
            pf = ParticleFilter(
                base_models=base_models,
                net=None,
                n_particles=N_PARTICLES,
                max_life=MAX_LIFE,
                use_net=False,
                const_noise=cn,
                const_prior=cp,
                const_lik=cl,
            ).eval()
            reps.append(tail_nll(pf, t_data, s_data))
    return float(np.mean(reps))

## Objective: leave-one-unit-out dev tail-NLL over all metrics

In [6]:
def objective(trial):
    cn = trial.suggest_float("const_noise", 0.05, 5.0, log=True)
    cl = trial.suggest_float("const_lik", 0.1, 10.0, log=True)
    cp = trial.suggest_float("const_prior", 0.0, 2.0)
    losses = []
    for name in perform_names:
        for u in dev_units:
            base = [dev_degmodels[name][v] for v in dev_units if v != u]
            seeds = [SEED + u * SEED_STRIDE + r for r in range(EVAL_REPS)]
            losses.append(eval_unit(base, dev_tensors[name][u], seeds, cn, cp, cl))
    return float(np.mean(losses))

## Run the study (parallel, resumable)

In [ ]:
storage = f"sqlite:///{(PRED_DIR / 'optuna_pf_gains.db').as_posix()}"
study = optuna.create_study(
    study_name=f"pf_gains_{DATA_NAME}",
    storage=storage,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True,
)

# N_TRIALS is the TOTAL target: run only the trials still missing in the db.
# Re-running with the same N_TRIALS just reports; a larger N_TRIALS tops up.
n_done = len(
    study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,))
)
n_remaining = max(0, N_TRIALS - n_done)
print(
    f"{n_done} completed trials in db; running {n_remaining} more to reach {N_TRIALS}"
)
if n_remaining:
    study.optimize(
        objective, n_trials=n_remaining, n_jobs=N_JOBS, show_progress_bar=True
    )

b = study.best_params
print("best value (mean tail-NLL):", study.best_value)
print("best params:", b)
(PRED_DIR / "optuna_best_gains.json").write_text(json.dumps(b, indent=2))

# ready-to-paste args entry (uses gains/make_args from pfnet_hparams)
print("\n# set the args0 baseline gains in src/training/pfnet_hparams.py:")
print(
    f'    0: make_args(net=network(hidden_dims=[32, 32], activation="tanh"), '
    f"gains_=gains("
    f"noise={round(b['const_noise'], 4)}, "
    f"prior={round(b['const_prior'], 4)}, "
    f"lik={round(b['const_lik'], 4)})),"
)


[I 2026-08-06 00:25:08,324] A new study created in RDB with name: pf_gains_DS03


0 completed trials in db; running 200 more to reach 200


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-08-06 00:43:24,669] Trial 2 finished with value: 39.49238060138844 and parameters: {'const_noise': 0.08738996975175829, 'const_lik': 0.9790943603918885, 'const_prior': 1.8617350881267725}. Best is trial 2 with value: 39.49238060138844.
[I 2026-08-06 00:43:26,873] Trial 3 finished with value: 45.68814397741247 and parameters: {'const_noise': 0.05712515699508433, 'const_lik': 0.732103323058822, 'const_prior': 0.4349783139423642}. Best is trial 2 with value: 39.49238060138844.
[I 2026-08-06 00:43:28,545] Trial 0 finished with value: 37.82068777746624 and parameters: {'const_noise': 0.06561654058852712, 'const_lik': 0.8073785684866485, 'const_prior': 0.20949292950468856}. Best is trial 0 with value: 37.82068777746624.
[I 2026-08-06 00:43:29,891] Trial 6 finished with value: 32.06618516533463 and parameters: {'const_noise': 0.1593417578935296, 'const_lik': 3.6180376587492318, 'const_prior': 1.9729995560441198}. Best is trial 6 with value: 32.06618516533463.
[I 2026-08-06 00:43:32,94

## Sanity check: tuned vs default gains

In [ ]:
def dev_score(cn, cp, cl):
    return float(
        np.mean(
            [
                eval_unit(
                    [dev_degmodels[n][v] for v in dev_units if v != u],
                    dev_tensors[n][u],
                    [SEED + u * SEED_STRIDE + r for r in range(EVAL_REPS)],
                    cn,
                    cp,
                    cl,
                )
                for n in perform_names
                for u in dev_units
            ]
        )
    )


b = study.best_params
print("default (1, 0, 1):", dev_score(1.0, 0.0, 1.0))
print(
    "tuned           :",
    dev_score(b["const_noise"], b["const_prior"], b["const_lik"]),
)
print("best params     :", b)